# Fine-Tuning T5 for Question Generation (FYP: AI-Centralized-LMS)

This notebook fine-tunes a T5 model on SQuAD to generate quiz questions from a passage + answer.
It replaces the Groq LLM-API quiz generator with a real, self-trained NLP model.

**Before running:** Go to `Runtime > Change runtime type` and set Hardware accelerator to **GPU (T4)**.

**Steps in this notebook:**
1. Install dependencies
2. Load and inspect the SQuAD dataset
3. Reformat data into `answer: ... context: ...` -> `question` pairs
4. Tokenize
5. Load base T5 model (baseline, before fine-tuning) and generate a sample question — this is your "before" comparison
6. Fine-tune
7. Evaluate with ROUGE and compare to the baseline
8. Save and download the fine-tuned model


## 1. Install dependencies

In [2]:
!pip install -q transformers datasets evaluate rouge_score accelerate sentencepiece


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


CUDA available: True
Device name: Tesla T4


## 2. Load the SQuAD dataset

SQuAD gives us (context, question, answer) triples — exactly what we need for question generation.
We're using SQuAD 1.1 here; each example has one answer per question, which keeps things simple.

In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("rajpurkar/squad")
print(raw_datasets)
print()
print("Example:")
print(raw_datasets["train"][0])


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Example:
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the

## 3. Reformat into QG input/target pairs

Input format: `answer: {answer_text} context: {context}`
Target: `{question}`

This is the standard format used by T5-based question generation models (e.g. `valhalla/t5-base-qg-hl`),
so you're recreating a well-established approach rather than inventing an untested one — good for your
FYP methodology section.

In [5]:
def format_example(example):
    answer_text = example["answers"]["text"][0] if example["answers"]["text"] else ""
    input_text = f"answer: {answer_text} context: {example['context']}"
    target_text = example["question"]
    return {"input_text": input_text, "target_text": target_text}

formatted = raw_datasets.map(format_example, remove_columns=raw_datasets["train"].column_names)
print(formatted["train"][0])


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'input_text': 'answer: Saint Bernadette Soubirous context: Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'target_text': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'}


## (Optional) Subsample for faster iteration

Full SQuAD train split is ~87k examples. On a free Colab T4, one full epoch over all of it
with t5-small takes a while. For your first run, subsampling to a smaller set lets you verify
the whole pipeline works end-to-end quickly. Once it works, come back and increase this (or
remove the subsampling entirely) for your final training run.

In [6]:
# Set to None to use the full dataset. Start with a subsample to sanity-check the pipeline.
TRAIN_SUBSET_SIZE = None   # try None later for full ~87k
VAL_SUBSET_SIZE = 2000

train_dataset = formatted["train"]
val_dataset = formatted["validation"]

if TRAIN_SUBSET_SIZE:
    train_dataset = train_dataset.shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_dataset = val_dataset.shuffle(seed=42).select(range(VAL_SUBSET_SIZE))

print("Train examples:", len(train_dataset))
print("Val examples:", len(val_dataset))


Train examples: 87599
Val examples: 2000


## 4. Tokenize

In [7]:
from transformers import T5TokenizerFast

MODEL_CHECKPOINT = "t5-small"  # switch to "t5-base" later if you want and Colab's GPU/time allows

tokenizer = T5TokenizerFast.from_pretrained(MODEL_CHECKPOINT)

MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 64

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels_ids
    ]
    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=["input_text", "target_text"])
tokenized_val = val_dataset.map(tokenize_batch, batched=True, remove_columns=["input_text", "target_text"])


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## 5. Baseline check — generate a question BEFORE fine-tuning

Run this now and save the output. You'll compare it against the same input after fine-tuning —
this before/after comparison is the strongest evidence in your report that fine-tuning actually
improved the model, rather than you just having downloaded and run someone else's checkpoint.

In [8]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

def generate_question(model, answer, context, max_length=64):
    input_text = f"answer: {answer} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_length=max_length, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

sample = raw_datasets["validation"][0]
sample_answer = sample["answers"]["text"][0]
sample_context = sample["context"]

print("Context:", sample_context[:300], "...")
print("Answer:", sample_answer)
print("Actual question:", sample["question"])
print()
baseline_question = generate_question(model, sample_answer, sample_context)
print("BASELINE (before fine-tuning) generated question:", baseline_question)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super B ...
Answer: Denver Broncos
Actual question: Which NFL team represented the AFC at Super Bowl 50?

BASELINE (before fine-tuning) generated question: The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title


## 6. Fine-tune

`fp16=True` gives a speed/memory boost on the T4 GPU Colab provides. If you see NaN losses,
turn it off. Training args below are reasonable defaults — feel free to adjust epochs/batch size
based on how much Colab time you have.

In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-qg-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,1.801340,1.661128
2,1.638696,1.617781


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.801340,1.661128
2,1.638696,1.617781
3,1.452973,1.618408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32850, training_loss=1.645632071995844, metrics={'train_runtime': 5235.7297, 'train_samples_per_second': 50.193, 'train_steps_per_second': 6.274, 'total_flos': 2.667556455107789e+16, 'train_loss': 1.645632071995844, 'epoch': 3.0})

## 7. Evaluate: ROUGE score + before/after comparison

In [10]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

metrics = trainer.evaluate(metric_key_prefix="eval")
print(metrics)


Training Loss,Validation Loss,Epoch
1.452973,1.618408,3


{'eval_loss': 1.6184083223342896}


In [ ]:
# Direct before/after comparison on the same example from step 5
finetuned_question = generate_question(model, sample_answer, sample_context)

print("Context:", sample_context[:300], "...")
print("Answer:", sample_answer)
print("Actual (reference) question:", sample["question"])
print()
print("BEFORE fine-tuning:", baseline_question)
print("AFTER fine-tuning: ", finetuned_question)


## 8. Try a few more examples for a qualitative feel

In [12]:
import random

random.seed(0)
for i in random.sample(range(len(raw_datasets["validation"])), 5):
    ex = raw_datasets["validation"][i]
    ans = ex["answers"]["text"][0]
    ctx = ex["context"]
    gen_q = generate_question(model, ans, ctx)
    print(f"Answer: {ans}")
    print(f"Reference question: {ex['question']}")
    print(f"Generated question:  {gen_q}")
    print("-" * 80)


Answer: his brutality
Reference question: What do some Mongolians feel non-Mongolian historians exaggerate about Genghis Khan?
Generated question:  What is there a chasm in the perception of Genghis Khan?
--------------------------------------------------------------------------------
Answer: lack of remorse
Reference question: Why is giving a defiant speech sometimes more harmful for the individual?
Generated question:  What did the U.S. Court of Appeals for the First Circuit suggest?
--------------------------------------------------------------------------------
Answer: Six
Reference question: How many Grammys has Lady Gaga won?
Generated question:  How many times was Lady Gaga a Grammy winner?
--------------------------------------------------------------------------------
Answer: 1997
Reference question: When did the UK formally subscribe to the Agreement on Social Policy?
Generated question:  When was the UK Labour Party elected to government?
------------------------------------

## 9. Save and download the fine-tuned model

This saves the model + tokenizer to a folder, zips it, and lets you download it directly from
Colab. Bring this zip back to your local machine and load it in Django with
`T5ForConditionalGeneration.from_pretrained("path/to/unzipped/folder")` — CPU inference with
t5-small is fast enough for a live demo.

In [13]:
SAVE_DIR = "t5-qg-finetuned"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -r t5-qg-finetuned.zip {SAVE_DIR}

from google.colab import files
files.download("t5-qg-finetuned.zip")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: t5-qg-finetuned/ (stored 0%)
  adding: t5-qg-finetuned/tokenizer.json (deflated 75%)
  adding: t5-qg-finetuned/tokenizer_config.json (deflated 82%)
  adding: t5-qg-finetuned/config.json (deflated 63%)
  adding: t5-qg-finetuned/model.safetensors (deflated 7%)
  adding: t5-qg-finetuned/generation_config.json (deflated 28%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Optional) Save to Google Drive instead

If the direct download is slow or the zip is large, mount your Drive and copy the folder there instead —
more reliable for larger models like t5-base.

In [14]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r t5-qg-finetuned /content/drive/MyDrive/t5-qg-finetuned
